# Unified_representation_geometry.ipynb

Unified separability, centroid displacement, directional alignment, and statistical exports for both open models.

Set BASE_DIR, MODEL_CONFIGS, and the CREMA-D paths, then run the required analysis sections in order.

Use the setup steps in the repository README before running this notebook.


In [ ]:
MOUNT_GOOGLE_DRIVE = True
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## Configuration

Set the model roots, output directory, layer selections, and sampling settings.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import platform
import re
import uuid

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from IPython.display import display
except ImportError:
    display = print

BASE_DIR = Path("/content/drive/MyDrive/subprobe")
OUTPUT_BASE = BASE_DIR / "geometry_unified"

MODEL_CONFIGS = [
    {"model": "Qwen2-Audio", "root": BASE_DIR / "task3_full_run_qwen",
     "layer": "lm_2", "suffix": ".pt"},
    {"model": "Audio-Flamingo-3", "root": BASE_DIR / "task3_full_run_flamingo",
     "layer": "lm_2", "suffix": ".pt"},
]

# Accepted analysis settings.
BASIC_LABELS = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
COARSE_MAP = {
    "anger": "ANG", "annoyance": "ANG", "disapproval": "ANG",
    "disgust": "DIS",
    "fear": "FEA", "nervousness": "FEA",
    "sadness": "SAD", "disappointment": "SAD", "remorse": "SAD", "grief": "SAD",
    "confusion": "SUR", "curiosity": "SUR", "realization": "SUR", "surprise": "SUR",
    "joy": "HAP", "amusement": "HAP", "excitement": "HAP", "pride": "HAP",
    "optimism": "HAP", "love": "HAP", "admiration": "HAP", "relief": "HAP",
    "gratitude": "HAP", "caring": "HAP", "desire": "HAP", "approval": "HAP",
    "neutral": "NEU", "embarrassment": "NEU",
}
COARSE_PRIORITY = ["ANG", "DIS", "FEA", "SAD", "SUR", "HAP", "NEU"]

# `unsure` is outside the paper's 28-emotion inventory. It is removed before
# grouping; rows left without an emotion are reported and excluded.
EXCLUDED_LABELS = {"unsure"}

MAX_SAMPLES_PER_SPLIT = 800
SAMPLE_SEED = 42
N_BOOTSTRAP = 5000
N_SEPARABILITY_PERMUTATIONS = 100
N_SHIFT_PERMUTATIONS = 100
PERMUTATION_SEED = 42

# This is the sole base-to-added shift analysis retained from probe_viz_*.
SHIFT_BASES = ["anger", "confusion", "neutral", "sadness", "surprise"]
MIN_CONDITION_SAMPLES = 1


## Load representations

Place `task3_results.csv` in each model root or its `layers/` directory. Set `rep_path` or store each representation under its sample ID. Use `.pt` dictionaries or `.npz` files.


In [ ]:
def parse_labels(value):
    if isinstance(value, list):
        labels = value
    else:
        try:
            labels = json.loads(value)
        except (TypeError, ValueError, json.JSONDecodeError):
            try:
                labels = ast.literal_eval(str(value))
            except (ValueError, SyntaxError) as exc:
                raise ValueError(f"Cannot parse labels: {value!r}") from exc
    if not isinstance(labels, list) or not all(isinstance(x, str) for x in labels):
        raise ValueError(f"Expected a list of emotion strings: {value!r}")
    return list(dict.fromkeys(x.strip().lower() for x in labels if x.strip().lower() not in EXCLUDED_LABELS))


def locate_metadata(root):
    candidates = [Path(root) / "task3_results.csv", Path(root) / "layers" / "task3_results.csv"]
    existing = [p for p in candidates if p.is_file()]
    if not existing:
        raise FileNotFoundError(f"No task3_results.csv at: {candidates}")
    return existing[0]


def locate_representation(row, config):
    suffix, root = config.get("suffix", ".pt"), Path(config["root"])
    candidates = []
    if "rep_path" in row and pd.notna(row["rep_path"]):
        stored = Path(str(row["rep_path"]).strip())
        candidates.append(stored)
        if not stored.is_absolute():
            candidates.append(root / stored)
    candidates += [root / "layers" / f"{row['id']}{suffix}", root / f"{row['id']}{suffix}"]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f"No representation for {config['model']} ID={row['id']}; tried {candidates}")


def raw_vector(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().float().numpy()
    x = np.asarray(value, dtype=np.float64)
    original_shape = tuple(x.shape)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x[0].mean(axis=0)
    elif x.ndim == 2 and x.shape[0] == 1:
        x = x[0]
    elif x.ndim != 1:
        raise ValueError(f"Expected [D], [1,D], or [1,T,D], found {original_shape}")
    if x.size == 0 or not np.isfinite(x).all() or np.linalg.norm(x) == 0:
        raise ValueError(f"Empty, non-finite, or zero representation; shape={original_shape}")
    return x


def load_layer(path, layer):
    if path.suffix == ".npz":
        with np.load(path, allow_pickle=False) as obj:
            if layer not in obj:
                raise KeyError(f"{path}: missing {layer}; keys={list(obj.keys())}")
            return raw_vector(obj[layer])
    try:
        import torch
    except ImportError as exc:
        raise ImportError("PyTorch is required for .pt inputs") from exc
    try:
        obj = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # PyTorch < 2.0
        obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict) and layer not in obj and isinstance(obj.get("reps"), dict):
        obj = obj["reps"]
    if not isinstance(obj, dict) or layer not in obj:
        keys = list(obj) if isinstance(obj, dict) else type(obj).__name__
        raise KeyError(f"{path}: missing {layer}; keys={keys}")
    return raw_vector(obj[layer])


def classify_split(labels):
    if len(labels) == 1 and labels[0] in BASIC_LABELS:
        return "basic"
    if len(labels) <= 1:
        return "fine_single"
    clusters = {COARSE_MAP[label] for label in labels if label in COARSE_MAP}
    clusters.discard("NEU")
    if len(clusters) == 0:
        return "fine_single"
    return "subtle_cross" if len(clusters) > 1 else "subtle_intra"


def coarse_target(labels):
    clusters = {COARSE_MAP[label] for label in labels if label in COARSE_MAP}
    for candidate in COARSE_PRIORITY:
        if candidate in clusters:
            return candidate
    return None


def load_model(config):
    metadata_path = locate_metadata(config["root"])
    frame = pd.read_csv(metadata_path, dtype={"id": str})
    if not {"id", "labels"}.issubset(frame.columns):
        raise ValueError(f"{metadata_path} requires id and labels columns")
    frame["id"] = frame["id"].str.strip()
    if frame["id"].duplicated().any():
        raise ValueError(f"{metadata_path}: duplicate IDs")
    frame["label_list"] = frame["labels"].apply(parse_labels)
    frame["exclusion_reason"] = np.where(frame["label_list"].map(len).eq(0), "no_retained_label", "")
    frame["split"] = frame["label_list"].apply(classify_split)
    frame["coarse_target"] = frame["label_list"].apply(coarse_target)
    usable = frame[frame["exclusion_reason"].eq("")].copy()
    vectors, paths = [], []
    for _, row in tqdm(usable.iterrows(), total=len(usable), desc=f"Loading {config['model']}"):
        path = locate_representation(row, config)
        try:
            vector = load_layer(path, config["layer"])
        except Exception as exc:
            raise RuntimeError(f"{config['model']} / ID={row['id']} / {config['layer']}: {exc}") from exc
        vectors.append(vector)
        paths.append(str(path))
    X = np.stack(vectors)
    usable["resolved_rep_path"] = paths
    usable["row_index"] = np.arange(len(usable))
    return {"config": config, "metadata_path": metadata_path, "frame": usable.reset_index(drop=True),
            "X": X, "excluded": frame[frame["exclusion_reason"].ne("")].copy()}


In [ ]:
def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


model_data = [load_model(config) for config in MODEL_CONFIGS]
inventory = pd.DataFrame([{ "Model": item["config"]["model"], "layer": item["config"]["layer"],
    "N_loaded": len(item["frame"]), "dimension": item["X"].shape[1],
    "excluded_empty": len(item["excluded"]), "metadata": str(item["metadata_path"]),
    "metadata_sha256": file_sha256(item["metadata_path"])}
    for item in model_data])
display(inventory)


## Nearest-centroid separability

Run the leave-one-out analysis for Basic, Intra, and Cross conditions with the configured coarse-label mapping and sampling settings.


In [ ]:
def unit_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    if np.any(~np.isfinite(norms)) or np.any(norms == 0):
        raise ValueError("Cannot cosine-normalize invalid vectors")
    return X / norms


def loo_nearest_centroid(X, labels):
    labels = np.asarray(labels)
    classes = np.array(sorted(set(labels)))
    class_to_index = {label: i for i, label in enumerate(classes)}
    encoded = np.array([class_to_index[label] for label in labels])
    counts = np.bincount(encoded, minlength=len(classes))
    sums = np.stack([X[encoded == j].sum(axis=0) for j in range(len(classes))])
    X_unit = unit_rows(X)
    centroid_unit = unit_rows(sums)
    similarities = X_unit @ centroid_unit.T
    for i, class_index in enumerate(encoded):
        if counts[class_index] <= 1:
            similarities[i, class_index] = -np.inf
        else:
            leave_one_sum = sums[class_index] - X[i]
            norm = np.linalg.norm(leave_one_sum)
            similarities[i, class_index] = (-np.inf if norm == 0 else
                                              float(X_unit[i] @ (leave_one_sum / norm)))
    predictions = classes[np.argmax(similarities, axis=1)]
    return predictions, predictions == labels


def bootstrap_mean_ci(correct, n_boot, seed):
    correct = np.asarray(correct, dtype=np.float64)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(correct), size=(n_boot, len(correct)))
    estimates = correct[indices].mean(axis=1)
    return np.quantile(estimates, [0.025, 0.975])


def separability_analysis(item):
    rng = np.random.default_rng(PERMUTATION_SEED)
    rows, predictions = [], []
    for split in ["basic", "subtle_intra", "subtle_cross"]:
        subset = item["frame"][item["frame"]["split"].eq(split)].copy()
        subset = subset.dropna(subset=["coarse_target"])
        if len(subset) > MAX_SAMPLES_PER_SPLIT:
            subset = subset.sample(MAX_SAMPLES_PER_SPLIT, random_state=SAMPLE_SEED)
        subset = subset.sort_values("id", kind="stable").reset_index(drop=True)
        if len(subset) < 2 or subset["coarse_target"].nunique() < 2:
            raise ValueError(f"{item['config']['model']} / {split}: insufficient examples/classes")
        X = item["X"][subset["row_index"].to_numpy()]
        y = subset["coarse_target"].to_numpy()
        pred, correct = loo_nearest_centroid(X, y)
        observed = float(correct.mean())
        lo, hi = bootstrap_mean_ci(correct, N_BOOTSTRAP,
                                   SAMPLE_SEED + ["basic", "subtle_intra", "subtle_cross"].index(split))
        null = np.empty(N_SEPARABILITY_PERMUTATIONS)
        for p in range(N_SEPARABILITY_PERMUTATIONS):
            _, perm_correct = loo_nearest_centroid(X, rng.permutation(y))
            null[p] = perm_correct.mean()
        p_value = (np.count_nonzero(null >= observed) + 1) / (len(null) + 1)
        rows.append({"Model": item["config"]["model"], "Split": split, "N": len(y),
                     "classes": len(set(y)), "SepAcc": observed, "CI_low": lo, "CI_high": hi,
                     "null_mean": null.mean(), "null_sd": null.std(ddof=1),
                     "p_value": p_value, "permutations": len(null)})
        predictions.extend({"Model": item["config"]["model"], "Split": split,
                            "id": sid, "target": target, "prediction": guess,
                            "correct": bool(ok)} for sid, target, guess, ok in
                           zip(subset["id"], y, pred, correct))
    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [ ]:
separability_parts, prediction_parts = zip(*(separability_analysis(item) for item in model_data))
separability_table = pd.concat(separability_parts, ignore_index=True)
separability_predictions = pd.concat(prediction_parts, ignore_index=True)
display(separability_table.round(4))


## Centroid displacement

Run the L2 base-to-added centroid-displacement analysis and shuffled-label comparisons for the configured emotion groups.


In [ ]:
def shift_condition_indices(label_lists):
    conditions = {}
    for base in SHIFT_BASES:
        conditions[(base, None)] = np.array([i for i, labels in enumerate(label_lists)
                                              if labels == [base]], dtype=int)
        for added in SHIFT_BASES:
            if added != base:
                conditions[(base, added)] = np.array([i for i, labels in enumerate(label_lists)
                                                       if base in labels and added in labels], dtype=int)
    return conditions


def shift_rows_from_assignment(X, conditions, assignment=None):
    assignment = np.arange(len(X)) if assignment is None else np.asarray(assignment)
    centroids, counts = {}, {}
    for condition, indices in conditions.items():
        if len(indices) >= MIN_CONDITION_SAMPLES:
            selected = assignment[indices]
            centroids[condition] = X[selected].mean(axis=0)
            counts[condition] = len(selected)
    rows = []
    for base in SHIFT_BASES:
        for added in SHIFT_BASES:
            if added == base or (base, None) not in centroids or (base, added) not in centroids:
                continue
            rows.append({"base": base, "added": added,
                         "magnitude": float(np.linalg.norm(centroids[(base, added)] - centroids[(base, None)])),
                         "base_N": counts[(base, None)], "pair_N": counts[(base, added)]})
    return pd.DataFrame(rows)


def directional_shift_analysis(item):
    label_lists = item["frame"]["label_list"].tolist()
    conditions = shift_condition_indices(label_lists)
    observed = shift_rows_from_assignment(item["X"], conditions)
    expected_pairs = len(SHIFT_BASES) * (len(SHIFT_BASES) - 1)
    if len(observed) != expected_pairs:
        missing = sorted(set((a, b) for a in SHIFT_BASES for b in SHIFT_BASES if a != b) -
                         set(zip(observed.get("base", []), observed.get("added", []))))
        raise ValueError(f"{item['config']['model']}: missing shift conditions {missing}")
    observed_mean = float(observed["magnitude"].mean())
    rng = np.random.default_rng(PERMUTATION_SEED)
    null = np.empty(N_SHIFT_PERMUTATIONS)
    for p in tqdm(range(N_SHIFT_PERMUTATIONS), desc=f"Shift null / {item['config']['model']}"):
        shuffled_assignment = rng.permutation(len(item["X"]))
        null[p] = shift_rows_from_assignment(item["X"], conditions, shuffled_assignment)["magnitude"].mean()
    summary = {"Model": item["config"]["model"], "mean_shift": observed_mean,
               "null_mean": float(null.mean()), "null_sd": float(null.std(ddof=1)),
               "p_value": (np.count_nonzero(null >= observed_mean) + 1) / (len(null) + 1),
               "permutations": len(null)}
    observed.insert(0, "Model", item["config"]["model"])
    null_frame = pd.DataFrame({"Model": item["config"]["model"],
                               "permutation": np.arange(len(null)), "mean_shift": null})
    return observed, summary, null_frame


In [ ]:
shift_outputs = [directional_shift_analysis(item) for item in model_data]
shift_table = pd.concat([x[0] for x in shift_outputs], ignore_index=True)
shift_summary = pd.DataFrame([x[1] for x in shift_outputs])
shift_null = pd.concat([x[2] for x in shift_outputs], ignore_index=True)
display(shift_summary.round(4))
display(shift_table.round(4))


## Figures and exports

Run this section to save the geometry figures, summary tables, and run metadata.


In [ ]:
def plot_shift_heatmaps(shift_table, configs, destination):
    import matplotlib.pyplot as plt
    import matplotlib.colors as colors

    with plt.rc_context({"font.size": 9, "axes.titlesize": 11, "axes.labelsize": 10,
                         "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"}):
        fig, axes = plt.subplots(1, len(configs), figsize=(4.7 * len(configs), 4.0), squeeze=False)
        for ax, config in zip(axes[0], configs):
            frame = shift_table[shift_table["Model"].eq(config["model"])]
            matrix = frame.pivot(index="base", columns="added", values="magnitude").reindex(
                index=SHIFT_BASES, columns=SHIFT_BASES)
            masked = np.ma.masked_invalid(matrix.to_numpy())
            image = ax.imshow(masked, cmap="YlOrRd", aspect="equal")
            for i in range(len(SHIFT_BASES)):
                for j in range(len(SHIFT_BASES)):
                    value = matrix.iloc[i, j]
                    if np.isfinite(value):
                        rgba = image.cmap(image.norm(value))
                        luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
                        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                                color="black" if luminance > 0.58 else "white", fontsize=8)
            ax.set_xticks(range(len(SHIFT_BASES)), [x.capitalize() for x in SHIFT_BASES], rotation=35, ha="right")
            ax.set_yticks(range(len(SHIFT_BASES)), [x.capitalize() for x in SHIFT_BASES])
            ax.set_xlabel("Added emotion")
            ax.set_ylabel("Base emotion")
            ax.set_title(config["model"])
            fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="L2 displacement")
        fig.tight_layout(w_pad=2.0)
        destination = Path(destination)
        for suffix in ["pdf", "svg", "png"]:
            fig.savefig(destination / f"centroid_displacement.{suffix}", dpi=300,
                        bbox_inches="tight", facecolor="white")
        return fig


def latex_escape(value):
    return str(value).replace("&", r"\&").replace("_", r"\_")


def separability_latex(frame):
    splits = ["basic", "subtle_intra", "subtle_cross"]
    lines = [r"\begin{table}[t]", r"\centering", r"\small", r"\begin{tabular}{lccc}",
             r"\hline", r"Model & Basic & Intra & Cross \\", r"\hline"]
    for model, group in frame.groupby("Model", sort=False):
        values = group.set_index("Split")
        cells = [f"{values.loc[s, 'SepAcc']:.3f}" for s in splits]
        lines.append(latex_escape(model) + " & " + " & ".join(cells) + r" \\")
    return "\n".join(lines + [r"\hline", r"\end{tabular}",
        r"\caption{Leave-one-out nearest-centroid separability.}",
        r"\label{tab:centroid-separability}", r"\end{table}", ""])


ACCEPTED_REFERENCE = {
    ("Qwen2-Audio", "separability/basic"): 0.471,
    ("Qwen2-Audio", "separability/subtle_intra"): 0.181,
    ("Qwen2-Audio", "separability/subtle_cross"): 0.211,
    ("Qwen2-Audio", "separability_null/basic"): 0.066,
    ("Qwen2-Audio", "separability_null/subtle_intra"): 0.069,
    ("Qwen2-Audio", "separability_null/subtle_cross"): 0.124,
    ("Audio-Flamingo-3", "separability/basic"): 0.875,
    ("Audio-Flamingo-3", "separability/subtle_intra"): 0.339,
    ("Audio-Flamingo-3", "separability/subtle_cross"): 0.400,
    ("Audio-Flamingo-3", "separability_null/basic"): 0.035,
    ("Audio-Flamingo-3", "separability_null/subtle_intra"): 0.064,
    ("Audio-Flamingo-3", "separability_null/subtle_cross"): 0.162,
    ("Qwen2-Audio", "mean_l2_shift"): 2.204061,
    ("Qwen2-Audio", "mean_l2_null"): 1.01,
    ("Audio-Flamingo-3", "mean_l2_shift"): 12.943237,
    ("Audio-Flamingo-3", "mean_l2_null"): 4.23,
}


def build_reconciliation(separability, shifts):
    computed = {}
    for row in separability.to_dict("records"):
        computed[(row["Model"], f"separability/{row['Split']}")] = row["SepAcc"]
        computed[(row["Model"], f"separability_null/{row['Split']}")] = row["null_mean"]
    for row in shifts.to_dict("records"):
        computed[(row["Model"], "mean_l2_shift")] = row["mean_shift"]
        computed[(row["Model"], "mean_l2_null")] = row["null_mean"]
    rows = []
    for key, accepted in ACCEPTED_REFERENCE.items():
        recomputed = computed.get(key, np.nan)
        rows.append({"Model": key[0], "Metric": key[1], "Accepted": accepted,
                     "Recomputed": recomputed, "Delta": recomputed - accepted})
    return pd.DataFrame(rows)


reconciliation = build_reconciliation(separability_table, shift_summary)
print("Accepted-paper reconciliation (review deltas before editing the manuscript)")
display(reconciliation.round(6))


stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_BASE / f"run_{stamp}_{uuid.uuid4().hex[:8]}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
inventory.to_csv(RUN_DIR / "input_inventory.csv", index=False)
separability_table.to_csv(RUN_DIR / "centroid_separability.csv", index=False)
separability_predictions.to_csv(RUN_DIR / "centroid_separability_predictions.csv", index=False)
shift_table.to_csv(RUN_DIR / "centroid_displacement.csv", index=False)
shift_summary.to_csv(RUN_DIR / "centroid_displacement_summary.csv", index=False)
shift_null.to_csv(RUN_DIR / "centroid_displacement_null.csv", index=False)
reconciliation.to_csv(RUN_DIR / "accepted_result_reconciliation.csv", index=False)
(RUN_DIR / "centroid_separability.tex").write_text(separability_latex(separability_table), encoding="utf-8")

figure = plot_shift_heatmaps(shift_table, MODEL_CONFIGS, RUN_DIR)
import matplotlib.pyplot as plt
plt.show()
plt.close(figure)

figure_tex = "\n".join([r"\begin{figure}[t]", r"\centering",
    r"\includegraphics[width=\linewidth]{centroid_displacement.pdf}",
    r"\caption{L2 displacement from single-label base-emotion centroids to centroids of "
    r"multilabel conditions containing the base and added emotions. Color scales are model-specific.}",
    r"\label{fig:centroid-displacement}", r"\end{figure}", ""])
(RUN_DIR / "centroid_displacement.tex").write_text(figure_tex, encoding="utf-8")

manifest = {"created_utc": stamp, "models": [{**x["config"], "root": str(x["config"]["root"]),
    "metadata": str(x["metadata_path"]), "N": len(x["frame"])} for x in model_data],
    "basic_labels": sorted(BASIC_LABELS), "coarse_map": COARSE_MAP,
    "coarse_priority": COARSE_PRIORITY, "excluded_labels": sorted(EXCLUDED_LABELS),
    "max_samples_per_split": MAX_SAMPLES_PER_SPLIT, "sample_seed": SAMPLE_SEED,
    "bootstrap": {"method": "resample fixed LOO correctness indicators", "iterations": N_BOOTSTRAP},
    "separability_permutations": N_SEPARABILITY_PERMUTATIONS,
    "shift_permutations": N_SHIFT_PERMUTATIONS, "permutation_seed": PERMUTATION_SEED,
    "shift_bases": SHIFT_BASES, "shift_metric": "L2 distance between raw pooled centroids",
    "python": platform.python_version()}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", RUN_DIR)


## Directional alignment

Configure the CREMA-D representations and run the following alignment and speaker-level statistics sections. Use the exported pair table for subsequent inference runs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import json
import uuid

import numpy as np
import pandas as pd

_required = {
    "model_data", "load_layer", "unit_rows",
    "OUTPUT_BASE", "COARSE_MAP", "file_sha256"
}
_missing = sorted(_required - set(globals()))
if _missing:
    raise RuntimeError(
        "Run the existing notebook's setup/loading cells first. "
        f"Missing variables: {_missing}"
    )

PROBING_BASE = Path("/content/drive/MyDrive/probing")

CREMAD_CONFIGS = [
    {
        "model": "Qwen2-Audio",
        "csv": PROBING_BASE / "full_probe" / "albe" / "merged_results.csv",
        "layer": "lm_2",
        "expected_n": 420,
    },
    {
        "model": "Audio-Flamingo-3",
        "csv": (
            PROBING_BASE
            / "full_probe_flamingo"
            / "abla"
            / "merged_results.csv"
        ),
        "layer": "lm_2",
        "expected_n": 420,
    },
]

TRANSITIONS = ["MD_TO_HI", "MD_TO_LO"]
COARSE_ORDER = ["ANG", "DIS", "FEA", "HAP", "SAD"]
EXPECTED_CELL_N = 42

CANONICAL = {
    "ANG": "anger",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "joy",
    "SAD": "sadness",
}

# Canonical labels are deliberately excluded from the subtle sets,
# matching the rebuttal analysis.
SUBTLE = {
    "ANG": ["annoyance", "disapproval"],
    "DIS": ["disapproval"],
    "FEA": ["nervousness"],
    "HAP": [
        "amusement", "excitement", "pride", "optimism", "love",
        "admiration", "relief", "gratitude", "caring", "desire",
    ],
    "SAD": ["disappointment", "remorse", "grief"],
}


def parse_retained_counts(value):
    if isinstance(value, dict):
        raw = value
    else:
        try:
            raw = json.loads(value)
        except (TypeError, ValueError):
            raw = ast.literal_eval(str(value))

    if not isinstance(raw, dict) or not raw:
        raise ValueError(f"Invalid label_counts: {value!r}")

    counts = {
        str(label).strip().lower(): float(count)
        for label, count in raw.items()
    }

    unknown = set(counts) - set(COARSE_MAP) - {"unsure"}
    invalid_values = any(
        not np.isfinite(count) or count < 0
        for count in counts.values()
    )
    if unknown or invalid_values:
        raise ValueError(f"Invalid label_counts: {value!r}")

    # Your requested rule:
    # remove unsure and renormalize using the remaining labels.
    return {
        label: count
        for label, count in counts.items()
        if label != "unsure" and count > 0
    }


def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError(
            "Cannot normalize an empty, non-finite, or zero vector"
        )
    return vector / norm


def weighted_centroid(
    item,
    frame,
    X_unit,
    label,
    require_maximal_support=False,
):
    indices = []
    weights = []

    for _, row in frame.iterrows():
        counts = row["_retained_counts"]

        if label not in counts:
            continue

        # Ties are retained, matching the accepted implementation.
        if (
            require_maximal_support
            and counts[label] < max(counts.values())
        ):
            continue

        indices.append(int(row["row_index"]))
        weights.append(counts[label] / sum(counts.values()))

    if not indices:
        raise ValueError(
            f"{item['config']['model']}: no EmoS support for {label}"
        )

    centroid = np.average(
        X_unit[np.asarray(indices)],
        axis=0,
        weights=np.asarray(weights),
    )
    return unit_vector(centroid), len(indices)


def build_emos_directions(item):
    frame = item["frame"].copy()

    if "label_counts" not in frame.columns:
        raise ValueError(
            f"{item['config']['model']}: label_counts is missing"
        )

    frame["_retained_counts"] = frame["label_counts"].apply(
        parse_retained_counts
    )

    # Explicitly exclude rows whose only annotation was unsure.
    frame = frame[
        frame["_retained_counts"].map(bool)
    ].copy()

    mismatch = frame.apply(
        lambda row: (
            set(row["label_list"])
            != set(row["_retained_counts"])
        ),
        axis=1,
    )
    if mismatch.any():
        examples = frame.loc[
            mismatch,
            ["id", "label_list", "_retained_counts"],
        ].head()
        raise ValueError(
            "Cleaned EmoS labels and label_counts disagree:\n"
            f"{examples}"
        )

    X_unit = unit_rows(item["X"])

    fine_labels = list(dict.fromkeys(
        label
        for coarse in COARSE_ORDER
        for label in SUBTLE[coarse]
    ))

    fine_centroids = {
        label: weighted_centroid(item, frame, X_unit, label)
        for label in fine_labels
    }

    directions = {}
    inventory = []

    for coarse in COARSE_ORDER:
        anchor, anchor_n = weighted_centroid(
            item,
            frame,
            X_unit,
            CANONICAL[coarse],
            require_maximal_support=True,
        )

        for subtle in SUBTLE[coarse]:
            subtle_centroid, subtle_n = fine_centroids[subtle]

            directions[(coarse, subtle)] = unit_vector(
                subtle_centroid - anchor
            )

            inventory.append({
                "Reference_dataset": "EmoS",
                "Model": item["config"]["model"],
                "Layer": item["config"]["layer"],
                "Coarse": coarse,
                "Canonical": CANONICAL[coarse],
                "Subtle": subtle,
                "Anchor_N": anchor_n,
                "Subtle_N": subtle_n,
            })

    return directions, pd.DataFrame(inventory)

In [ ]:
def parse_cremad_filename(value):
    filename = Path(
        str(value).strip().replace("\\", "/")
    ).name
    parts = Path(filename).stem.split("_")

    if len(parts) < 4:
        raise ValueError(f"Invalid CREMA-D filename: {value!r}")

    return {
        "speaker": parts[0],
        "utterance": parts[1],
        "coarse": parts[2].upper(),
        "intensity": parts[3].upper(),
    }


def resolve_cremad_representation(value, csv_path):
    normalized = str(value).strip().replace("\\", "/")
    raw = Path(normalized)

    candidates = [raw]
    if not raw.is_absolute():
        candidates.extend([
            Path(csv_path).parent / raw,
            PROBING_BASE / raw,
        ])

    # Remap an older absolute path onto the current Colab probing root.
    marker = "/probing/"
    folded = normalized.casefold()
    if marker in folded:
        relative = normalized[
            folded.rfind(marker) + len(marker):
        ]
        candidates.append(PROBING_BASE / relative)

    tried = []
    for candidate in candidates:
        candidate = Path(candidate)
        if str(candidate) in tried:
            continue
        tried.append(str(candidate))
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Cannot resolve {value!r}; tried {tried}"
    )


def load_cremad_pairs(config):
    csv_path = Path(config["csv"])
    frame = pd.read_csv(csv_path, dtype={"id": str})

    required = {
        "id",
        "task",
        "neutral_path",
        "manipulated_path",
        "neutral_rep_path",
        "rep_path",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{csv_path}: missing columns {missing}")

    frame = frame[
        frame["task"]
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("intensity")
    ].copy()

    rows = []

    for _, row in frame.iterrows():
        source = parse_cremad_filename(row["neutral_path"])
        target = parse_cremad_filename(row["manipulated_path"])

        for key in ["speaker", "utterance", "coarse"]:
            if source[key] != target[key]:
                raise ValueError(
                    f"Unpaired CREMA-D row: id={row['id']}"
                )

        transition = (
            f"{source['intensity']}_TO_{target['intensity']}"
        )

        if (
            source["coarse"] not in COARSE_ORDER
            or transition not in TRANSITIONS
        ):
            continue

        rows.append({
            "Dataset": "CREMA-D",
            "Model": config["model"],
            "id": row["id"],
            "speaker": source["speaker"],
            "utterance": source["utterance"],
            "Coarse": source["coarse"],
            "Transition": transition,
            "source_rep": str(resolve_cremad_representation(
                row["neutral_rep_path"], csv_path
            )),
            "target_rep": str(resolve_cremad_representation(
                row["rep_path"], csv_path
            )),
        })

    if not rows:
        raise ValueError(
            f"{csv_path}: no retained MD→HI/MD→LO pairs"
        )

    pairs = pd.DataFrame(rows)

    pair_key = [
        "speaker", "utterance", "Coarse", "Transition"
    ]
    if pairs.duplicated(pair_key).any():
        raise ValueError(
            f"{csv_path}: duplicate CREMA-D transition pairs"
        )

    if len(pairs) != config["expected_n"]:
        raise ValueError(
            f"{csv_path}: expected {config['expected_n']} pairs, "
            f"found {len(pairs)}"
        )

    expected_index = pd.MultiIndex.from_product(
        [COARSE_ORDER, TRANSITIONS],
        names=["Coarse", "Transition"],
    )
    counts = (
        pairs.groupby(["Coarse", "Transition"])
        .size()
        .reindex(expected_index, fill_value=0)
    )

    if not counts.eq(EXPECTED_CELL_N).all():
        raise ValueError(
            f"{csv_path}: expected {EXPECTED_CELL_N} pairs per "
            f"emotion/transition cell; got {counts.to_dict()}"
        )

    return pairs, csv_path


def align_cremad_pairs(item, pairs, layer):
    if item["config"]["layer"] != layer:
        raise ValueError(
            f"Layer mismatch: EmoS={item['config']['layer']}, "
            f"CREMA-D={layer}"
        )

    directions, centroid_inventory = build_emos_directions(item)
    expected_dimension = item["X"].shape[1]

    representation_cache = {}
    pair_rows = []
    detail_rows = []

    def representation(path):
        key = (str(path), layer)

        if key not in representation_cache:
            vector = unit_vector(
                load_layer(Path(path), layer)
            )

            if vector.shape != (expected_dimension,):
                raise ValueError(
                    f"{path}: expected dimension "
                    f"{expected_dimension}, found {vector.shape}"
                )

            representation_cache[key] = vector

        return representation_cache[key]

    iterator = tqdm(
        pairs.iterrows(),
        total=len(pairs),
        desc=f"CREMA-D alignment / {item['config']['model']}",
    )

    for _, row in iterator:
        # Match the rebuttal implementation: normalize source and
        # target individually, subtract, and normalize the displacement.
        displacement = unit_vector(
            representation(row["target_rep"])
            - representation(row["source_rep"])
        )

        similarities = []

        for subtle in SUBTLE[row["Coarse"]]:
            similarity = float(
                displacement
                @ directions[(row["Coarse"], subtle)]
            )
            similarities.append(similarity)

            detail_rows.append({
                "Reference_dataset": "EmoS",
                "Shift_dataset": "CREMA-D",
                "Model": item["config"]["model"],
                "Layer": layer,
                "id": row["id"],
                "speaker": row["speaker"],
                "utterance": row["utterance"],
                "Coarse": row["Coarse"],
                "Transition": row["Transition"],
                "Subtle": subtle,
                "directional_alignment": similarity,
            })

        pair_rows.append({
            "Reference_dataset": "EmoS",
            "Shift_dataset": "CREMA-D",
            "Model": item["config"]["model"],
            "Layer": layer,
            "id": row["id"],
            "speaker": row["speaker"],
            "utterance": row["utterance"],
            "Coarse": row["Coarse"],
            "Transition": row["Transition"],
            "N_subtle_directions": len(similarities),
            "mean_subtle_alignment": float(
                np.mean(similarities)
            ),
        })

    return (
        pd.DataFrame(pair_rows),
        pd.DataFrame(detail_rows),
        centroid_inventory,
    )

In [ ]:
model_by_name = {
    item["config"]["model"]: item
    for item in model_data
}

pair_inputs = []
pair_outputs = []
detail_outputs = []
centroid_outputs = []
resolved_cremad_csvs = {}

for config in CREMAD_CONFIGS:
    model = config["model"]

    if model not in model_by_name:
        raise ValueError(
            f"No loaded EmoS representations for {model}"
        )

    pairs, resolved_csv = load_cremad_pairs(config)

    pair_result, detail_result, centroid_result = (
        align_cremad_pairs(
            model_by_name[model],
            pairs,
            config["layer"],
        )
    )

    pair_inputs.append(pairs)
    pair_outputs.append(pair_result)
    detail_outputs.append(detail_result)
    centroid_outputs.append(centroid_result)
    resolved_cremad_csvs[model] = resolved_csv

cremad_pairs = pd.concat(
    pair_inputs, ignore_index=True
)
directional_pairs = pd.concat(
    pair_outputs, ignore_index=True
)
directional_details = pd.concat(
    detail_outputs, ignore_index=True
)
direction_centroid_inventory = pd.concat(
    centroid_outputs, ignore_index=True
)

directional_summary = (
    directional_pairs
    .groupby(
        [
            "Reference_dataset",
            "Shift_dataset",
            "Model",
            "Layer",
            "Coarse",
            "Transition",
        ],
        sort=False,
    )["mean_subtle_alignment"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "mean": "mean_subtle_alignment",
        "std": "sd",
        "count": "N_pairs",
    })
)

transition_inventory = (
    cremad_pairs
    .groupby(["Dataset", "Model", "Coarse", "Transition"])
    .size()
    .rename("N_pairs")
    .reset_index()
)

# Values saved in the accepted/rebuttal notebook.
ACCEPTED_DIRECTIONAL_VALUES = {
    ("Qwen2-Audio", "ANG", "MD_TO_HI"): -0.003434,
    ("Qwen2-Audio", "ANG", "MD_TO_LO"):  0.019036,
    ("Qwen2-Audio", "DIS", "MD_TO_HI"): -0.005701,
    ("Qwen2-Audio", "DIS", "MD_TO_LO"):  0.006187,
    ("Qwen2-Audio", "FEA", "MD_TO_HI"): -0.042748,
    ("Qwen2-Audio", "FEA", "MD_TO_LO"):  0.023806,
    ("Qwen2-Audio", "HAP", "MD_TO_HI"): -0.018841,
    ("Qwen2-Audio", "HAP", "MD_TO_LO"):  0.017205,
    ("Qwen2-Audio", "SAD", "MD_TO_HI"): -0.000456,
    ("Qwen2-Audio", "SAD", "MD_TO_LO"):  0.002396,

    ("Audio-Flamingo-3", "ANG", "MD_TO_HI"): -0.052209,
    ("Audio-Flamingo-3", "ANG", "MD_TO_LO"):  0.206540,
    ("Audio-Flamingo-3", "DIS", "MD_TO_HI"):  0.007451,
    ("Audio-Flamingo-3", "DIS", "MD_TO_LO"): -0.016234,
    ("Audio-Flamingo-3", "FEA", "MD_TO_HI"): -0.183759,
    ("Audio-Flamingo-3", "FEA", "MD_TO_LO"):  0.128662,
    ("Audio-Flamingo-3", "HAP", "MD_TO_HI"): -0.015660,
    ("Audio-Flamingo-3", "HAP", "MD_TO_LO"):  0.019948,
    ("Audio-Flamingo-3", "SAD", "MD_TO_HI"):  0.014783,
    ("Audio-Flamingo-3", "SAD", "MD_TO_LO"): -0.004221,
}

accepted = pd.DataFrame([
    {
        "Model": model,
        "Coarse": coarse,
        "Transition": transition,
        "Accepted": value,
    }
    for (model, coarse, transition), value
    in ACCEPTED_DIRECTIONAL_VALUES.items()
])

directional_reconciliation = (
    accepted.merge(
        directional_summary[
            [
                "Model",
                "Coarse",
                "Transition",
                "mean_subtle_alignment",
            ]
        ],
        on=["Model", "Coarse", "Transition"],
        how="left",
        validate="one_to_one",
    )
    .rename(columns={
        "mean_subtle_alignment": "Recomputed"
    })
)

directional_reconciliation["Delta"] = (
    directional_reconciliation["Recomputed"]
    - directional_reconciliation["Accepted"]
)
directional_reconciliation["Status"] = np.where(
    directional_reconciliation["Delta"].abs() <= 5e-6,
    "MATCH",
    "CHECK",
)

display(transition_inventory)
display(directional_summary.round(6))

print(
    "Reconciliation against the rebuttal values "
    "(do not silently replace manuscript values):"
)
display(directional_reconciliation.round(6))

In [ ]:
import matplotlib.pyplot as plt


def plot_cremad_directional_alignment(summary, destination):
    colors = {
        "MD_TO_HI": "#D55E00",
        "MD_TO_LO": "#0072B2",
    }
    transition_labels = {
        "MD_TO_HI": "MD→HI",
        "MD_TO_LO": "MD→LO",
    }
    coarse_labels = {
        "ANG": "Anger",
        "DIS": "Disgust",
        "FEA": "Fear",
        "HAP": "Happiness",
        "SAD": "Sadness",
    }

    models = [
        config["model"]
        for config in CREMAD_CONFIGS
    ]

    style = {
        "font.size": 9,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }

    with plt.rc_context(style):
        fig, axes = plt.subplots(
            1,
            len(models),
            figsize=(4.8 * len(models), 3.5),
            sharey=True,
            squeeze=False,
        )

        x = np.arange(len(COARSE_ORDER))
        width = 0.36

        for ax, model in zip(axes[0], models):
            matrix = (
                summary[summary["Model"].eq(model)]
                .pivot(
                    index="Coarse",
                    columns="Transition",
                    values="mean_subtle_alignment",
                )
                .reindex(
                    index=COARSE_ORDER,
                    columns=TRANSITIONS,
                )
            )

            if matrix.isna().any().any():
                raise ValueError(
                    f"Incomplete figure matrix for {model}"
                )

            for offset, transition in zip(
                [-width / 2, width / 2],
                TRANSITIONS,
            ):
                ax.bar(
                    x + offset,
                    matrix[transition],
                    width,
                    label=transition_labels[transition],
                    color=colors[transition],
                )

            ax.axhline(
                0,
                color="#444444",
                linewidth=0.9,
            )
            ax.set_xticks(
                x,
                [coarse_labels[c] for c in COARSE_ORDER],
                rotation=25,
                ha="right",
            )
            ax.set_xlabel("CREMA-D coarse emotion")
            ax.set_title(model)
            ax.grid(
                axis="y",
                alpha=0.22,
                linewidth=0.7,
            )
            ax.set_axisbelow(True)

        axes[0, 0].set_ylabel(
            "Cosine alignment with EmoS\n"
            "coarse→subtle directions"
        )

        handles, labels = (
            axes[0, 0].get_legend_handles_labels()
        )
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=2,
            frameon=False,
            bbox_to_anchor=(0.5, 1.02),
        )
        fig.tight_layout(
            rect=[0, 0, 1, 0.94],
            w_pad=1.4,
        )

        destination = Path(destination)
        for suffix in ["pdf", "svg", "png"]:
            fig.savefig(
                destination
                / f"cremad_directional_alignment.{suffix}",
                dpi=300,
                bbox_inches="tight",
                facecolor="white",
            )

        return fig


if (
    "RUN_DIR" in globals()
    and Path(RUN_DIR).is_dir()
):
    DIRECTION_RUN_DIR = Path(RUN_DIR)
else:
    stamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%SZ")
    DIRECTION_RUN_DIR = (
        OUTPUT_BASE
        / f"directional_alignment_{stamp}_{uuid.uuid4().hex[:8]}"
    )
    DIRECTION_RUN_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

cremad_pairs.to_csv(
    DIRECTION_RUN_DIR / "cremad_directional_pairs.csv",
    index=False,
)
transition_inventory.to_csv(
    DIRECTION_RUN_DIR / "cremad_directional_inventory.csv",
    index=False,
)
direction_centroid_inventory.to_csv(
    DIRECTION_RUN_DIR
    / "emos_direction_centroid_inventory.csv",
    index=False,
)
directional_pairs.to_csv(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_pairs.csv",
    index=False,
)
directional_details.to_csv(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_per_direction.csv",
    index=False,
)
directional_summary.to_csv(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_summary.csv",
    index=False,
)
directional_reconciliation.to_csv(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_reconciliation.csv",
    index=False,
)

directional_figure = plot_cremad_directional_alignment(
    directional_summary,
    DIRECTION_RUN_DIR,
)
plt.show()
plt.close(directional_figure)

figure_tex = "\n".join([
    r"\begin{figure}[t]",
    r"\centering",
    r"\includegraphics[width=\linewidth]{cremad_directional_alignment.pdf}",
    r"\caption{Directional alignment between CREMA-D intensity-induced "
    r"representation shifts and EmoS-derived coarse-to-subtle emotion "
    r"directions. Positive values indicate similarly oriented displacement; "
    r"they do not establish reduced distance to a subtle centroid.}",
    r"\label{fig:change-shift}",
    r"\end{figure}",
    "",
])
(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment.tex"
).write_text(figure_tex, encoding="utf-8")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "reference_dataset": "EmoS",
    "shift_dataset": "CREMA-D",
    "layer_by_model": {
        config["model"]: config["layer"]
        for config in CREMAD_CONFIGS
    },
    "source_csvs": {
        model: {
            "path": str(path),
            "sha256": file_sha256(path),
        }
        for model, path in resolved_cremad_csvs.items()
    },
    "transitions": TRANSITIONS,
    "coarse_order": COARSE_ORDER,
    "expected_pairs_per_model": 420,
    "expected_pairs_per_cell": EXPECTED_CELL_N,
    "normalization": (
        "unit-normalize each sample before centroid "
        "and displacement construction"
    ),
    "pair_aggregation": (
        "mean cosine over mapped fine-grained directions"
    ),
    "unsure_handling": (
        "remove unsure; renormalize over retained labels; "
        "exclude rows with no retained label"
    ),
}
(
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_manifest.json"
).write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(
    "Saved CREMA-D directional-alignment outputs:",
    DIRECTION_RUN_DIR,
)

In [ ]:
# =========================================================
# WITHIN-CREMA-D CENTROID CONTROL — COMPLETED RESULTS
# Reproduces geometry.ipynb cells 12–15 without reloading .pt files.
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

within_cremad_table = pd.DataFrame([
    ["Qwen2-Audio",      "Anger",     0.0000,  0.0010],
    ["Qwen2-Audio",      "Disgust",  -0.0001, -0.0005],
    ["Qwen2-Audio",      "Fear",      0.0017, -0.0006],
    ["Qwen2-Audio",      "Happiness", 0.0148, -0.0000],
    ["Qwen2-Audio",      "Sadness",  -0.0010, -0.0006],
    ["Audio-Flamingo-3", "Anger",     0.0720, -0.2339],
    ["Audio-Flamingo-3", "Disgust",  -0.0210,  0.0328],
    ["Audio-Flamingo-3", "Fear",      0.2509, -0.1284],
    ["Audio-Flamingo-3", "Happiness", 0.0497, -0.0323],
    ["Audio-Flamingo-3", "Sadness",  -0.1246,  0.0337],
], columns=["Model", "Emotion", "MD_TO_HI", "MD_TO_LO"])

display(within_cremad_table.round(4))

models = ["Qwen2-Audio", "Audio-Flamingo-3"]
emotions = ["Anger", "Disgust", "Fear", "Happiness", "Sadness"]
colors = {"MD_TO_HI": "#D55E00", "MD_TO_LO": "#0072B2"}
labels = {"MD_TO_HI": "MD→HI", "MD_TO_LO": "MD→LO"}

with plt.rc_context({
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}):
    fig, axes = plt.subplots(
        1, 2, figsize=(9.6, 3.5), sharey=True
    )

    x = np.arange(len(emotions))
    width = 0.36

    for ax, model in zip(axes, models):
        frame = (
            within_cremad_table[
                within_cremad_table["Model"].eq(model)
            ]
            .set_index("Emotion")
            .reindex(emotions)
        )

        for offset, transition in zip(
            (-width / 2, width / 2),
            ["MD_TO_HI", "MD_TO_LO"],
        ):
            ax.bar(
                x + offset,
                frame[transition],
                width,
                color=colors[transition],
                label=labels[transition],
            )

        ax.axhline(0, color="#444444", linewidth=0.9)
        ax.set_xticks(x, emotions, rotation=25, ha="right")
        ax.set_xlabel("CREMA-D emotion")
        ax.set_title(model)
        ax.grid(axis="y", alpha=0.22, linewidth=0.7)
        ax.set_axisbelow(True)

    axes[0].set_ylabel(
        "Change in cosine similarity\n"
        "to CREMA-D emotion centroid"
    )
    axes[0].set_ylim(-0.27, 0.29)

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        legend_labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02),
    )

    fig.tight_layout(rect=(0, 0, 1, 0.94), w_pad=1.4)

    output_dir = (
        Path(RUN_DIR)
        if "RUN_DIR" in globals()
        else Path.cwd()
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    within_cremad_table.to_csv(
        output_dir / "within_cremad_centroid_control.csv",
        index=False,
    )

    for suffix in ["pdf", "svg", "png"]:
        fig.savefig(
            output_dir / f"within_cremad_centroid_control.{suffix}",
            dpi=300,
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

# Run after the CREMA-D alignment cells, or set this to the previously exported
# cremad_directional_alignment_pairs.csv and run only this new section.
ALIGNMENT_PAIRS_CSV = "/content/drive/MyDrive/subprobe/geometry_unified/run_20260913T133917Z_c79a33ce/cremad_directional_alignment_pairs.csv"
ALIGNMENT_N_SIGN_FLIPS = 19_999
ALIGNMENT_N_BOOTSTRAP = 9_999
ALIGNMENT_MAX_EXACT_SPEAKERS = 16
ALIGNMENT_STATS_SEED = 20261002
ALIGNMENT_ALPHA = 0.05

ALIGNMENT_GROUP_COLUMNS = [
    "Reference_dataset", "Shift_dataset", "Model", "Layer", "Coarse"
]
ALIGNMENT_TRANSITIONS = ["MD_TO_HI", "MD_TO_LO"]


def alignment_seed(test_key, purpose, seed):
    """Stable, separate RNG streams for every hypothesis and resampling method."""
    payload = json.dumps([int(seed), list(test_key), purpose], ensure_ascii=True)
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little")


def validate_alignment_pairs(frame):
    required = ALIGNMENT_GROUP_COLUMNS + [
        "Transition", "speaker", "utterance", "mean_subtle_alignment"
    ]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Alignment-pair table is missing columns: {missing}")
    frame = frame[required].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError("Alignment-pair table must be nonempty with no missing values")
    string_columns = [c for c in required if c != "mean_subtle_alignment"]
    for column in string_columns:
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"Empty value in {column}")
    frame["mean_subtle_alignment"] = pd.to_numeric(
        frame["mean_subtle_alignment"], errors="raise"
    ).astype(float)
    scores = frame["mean_subtle_alignment"].to_numpy()
    if not np.isfinite(scores).all() or np.any(np.abs(scores) > 1 + 1e-12):
        raise ValueError("Mean cosine alignment must be finite and in [-1, 1]")
    if set(frame["Transition"]) != set(ALIGNMENT_TRANSITIONS):
        raise ValueError("Expected exactly MD_TO_HI and MD_TO_LO transitions")
    pair_key = ALIGNMENT_GROUP_COLUMNS + ["speaker", "utterance", "Transition"]
    if frame.duplicated(pair_key).any():
        raise ValueError("Duplicate model/emotion/speaker/utterance/transition rows")
    return frame.sort_values(pair_key, kind="stable").reset_index(drop=True)


def holm_adjust(p_values):
    """Holm family-wise correction, valid without independence between tests."""
    p_values = np.asarray(p_values, dtype=np.float64)
    if p_values.ndim != 1 or len(p_values) == 0:
        raise ValueError("Expected a nonempty one-dimensional p-value array")
    if not np.isfinite(p_values).all() or np.any((p_values < 0) | (p_values > 1)):
        raise ValueError("Invalid p-value")
    order = np.argsort(p_values, kind="stable")
    adjusted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    result = np.empty_like(p_values)
    result[order] = np.minimum(adjusted, 1.0)
    return result


def speaker_cluster_inference(
    values, speakers, test_key, *,
    n_sign_flips=ALIGNMENT_N_SIGN_FLIPS,
    n_bootstrap=ALIGNMENT_N_BOOTSTRAP,
    max_exact_speakers=ALIGNMENT_MAX_EXACT_SPEAKERS,
    seed=ALIGNMENT_STATS_SEED,
):
    """Two-sided speaker sign flips plus percentile speaker-bootstrap CI.

    The statistic remains the pair-weighted mean. Every row of a speaker
    receives the same random sign, and bootstrap draws retain whole speakers.
    Sign-flip validity requires independent speaker clusters whose score
    vectors are jointly sign-symmetric under H0; a zero mean alone is not enough.
    """
    values = np.asarray(values, dtype=np.float64)
    speakers = np.asarray(speakers)
    if values.ndim != 1 or speakers.ndim != 1 or len(values) != len(speakers):
        raise ValueError("Values and speakers must be equal-length vectors")
    if len(values) == 0 or not np.isfinite(values).all() or pd.isna(speakers).any():
        raise ValueError("Values and speakers must be nonempty and finite/nonmissing")
    if n_sign_flips < 1 or n_bootstrap < 2 or not 0 <= max_exact_speakers <= 20:
        raise ValueError("Invalid resampling counts or exact-enumeration limit")

    cluster_input = pd.DataFrame({"speaker": speakers.astype(str), "value": values})
    # Stable ordering also makes results invariant to input-row ordering.
    cluster_input = cluster_input.sort_values(["speaker", "value"], kind="stable")
    clusters = cluster_input.groupby("speaker", sort=True)["value"].agg(["sum", "count"])
    clusters = clusters.rename(columns={"sum": "score_sum", "count": "N_pairs"}).reset_index()
    n_speakers = len(clusters)
    if n_speakers < 2:
        raise ValueError(f"{test_key}: at least two independent speakers are required")

    sums = clusters["score_sum"].to_numpy(dtype=np.float64)
    counts = clusters["N_pairs"].to_numpy(dtype=np.int64)
    denominator = int(counts.sum())
    observed = float(sums.sum() / denominator)
    # Handle floating-point ties relative to the arithmetic scale, including
    # cancellation near zero. This cannot turn a null p-value into zero.
    tolerance = 100 * np.finfo(np.float64).eps * max(
        float(np.abs(sums).sum() / denominator), np.finfo(np.float64).tiny
    )

    exact = n_speakers <= max_exact_speakers
    n_draws = (1 << n_speakers) if exact else int(n_sign_flips)
    flip_seed = alignment_seed(test_key, "speaker_sign_flip", seed)
    flip_rng = np.random.default_rng(flip_seed)
    extreme = 0
    for start in range(0, n_draws, 1024):
        batch = min(1024, n_draws - start)
        if exact:
            patterns = np.arange(start, start + batch, dtype=np.uint64)[:, None]
            bits = (patterns >> np.arange(n_speakers, dtype=np.uint64)) & np.uint64(1)
            signs = 2 * bits.astype(np.int8) - 1
        else:
            signs = 2 * flip_rng.integers(0, 2, size=(batch, n_speakers), dtype=np.int8) - 1
        null_means = (signs * sums[None, :]).sum(axis=1) / denominator
        extreme += int(np.count_nonzero(np.abs(null_means) >= abs(observed) - tolerance))

    # Exact enumeration includes the observed sign pattern; do not add one.
    # Random Monte Carlo draws use the finite-resampling +1 correction.
    p_value = extreme / n_draws if exact else (extreme + 1) / (n_draws + 1)

    bootstrap_seed = alignment_seed(test_key, "speaker_bootstrap", seed)
    bootstrap_rng = np.random.default_rng(bootstrap_seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)
    for start in range(0, n_bootstrap, 1024):
        batch = min(1024, n_bootstrap - start)
        draw = bootstrap_rng.integers(0, n_speakers, size=(batch, n_speakers))
        bootstrap_means[start:start + batch] = (
            sums[draw].sum(axis=1) / counts[draw].sum(axis=1)
        )
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    clusters["speaker_mean"] = clusters["score_sum"] / clusters["N_pairs"]

    result = {
        "estimate": observed,
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "p_raw": float(p_value),
        "N_pairs": denominator,
        "N_speakers": n_speakers,
        "min_pairs_per_speaker": int(counts.min()),
        "max_pairs_per_speaker": int(counts.max()),
        "test_method": "exact_speaker_sign_flip" if exact else "monte_carlo_speaker_sign_flip",
        "alternative": "two-sided",
        "null_value": 0.0,
        "sign_patterns_evaluated": n_draws,
        "extreme_patterns": extreme,
        "bootstrap_draws": int(n_bootstrap),
        "sign_flip_seed": flip_seed,
        "bootstrap_seed": bootstrap_seed,
        "fewer_than_10_speakers": bool(n_speakers < 10),
    }
    return result, clusters


def directional_alignment_inference(frame, *, alpha=ALIGNMENT_ALPHA, **resampling):
    """Test each transition vs zero and each matched HI-minus-LO contrast.

    All returned hypotheses belong to ONE Holm family (30 for the current
    two-model, five-emotion notebook). Pair scores are already averaged across
    subtle directions; per-direction rows must not be passed as replicates.
    """
    if not 0 < alpha < 1:
        raise ValueError("alpha must be between zero and one")
    frame = validate_alignment_pairs(frame)
    match_key = ALIGNMENT_GROUP_COLUMNS + ["speaker", "utterance"]
    matched = frame.pivot(
        index=match_key, columns="Transition", values="mean_subtle_alignment"
    ).reindex(columns=ALIGNMENT_TRANSITIONS)
    if matched.isna().any().any():
        missing = matched[matched.isna().any(axis=1)].reset_index().head().to_dict("records")
        raise ValueError(f"Unmatched MD_TO_HI/MD_TO_LO rows; no rows were dropped: {missing}")
    matched.columns.name = None
    matched = matched.reset_index()
    matched["HI_minus_LO"] = matched["MD_TO_HI"] - matched["MD_TO_LO"]

    results, speaker_tables = [], []

    def run_group(group, keys, test, transition, score_column):
        meta = dict(zip(ALIGNMENT_GROUP_COLUMNS, keys))
        meta.update({"Test": test, "Transition": transition})
        result, speakers = speaker_cluster_inference(
            group[score_column].to_numpy(),
            group["speaker"].to_numpy(),
            tuple(keys) + (test, transition),
            **resampling,
        )
        results.append({**meta, **result})
        for name, value in meta.items():
            speakers[name] = value
        speaker_tables.append(speakers)

    for keys, group in frame.groupby(ALIGNMENT_GROUP_COLUMNS + ["Transition"], sort=True):
        run_group(group, keys[:-1], "alignment_vs_zero", keys[-1], "mean_subtle_alignment")
    for keys, group in matched.groupby(ALIGNMENT_GROUP_COLUMNS, sort=True):
        run_group(group, keys, "paired_HI_minus_LO", "MD_TO_HI_minus_MD_TO_LO", "HI_minus_LO")

    results = pd.DataFrame(results)
    results["p_holm"] = holm_adjust(results["p_raw"].to_numpy())
    results["holm_family_size"] = len(results)
    results["reject_holm_0_05"] = results["p_holm"] <= alpha
    # The column is explicitly named by the configured threshold below.
    if alpha != 0.05:
        results = results.rename(columns={"reject_holm_0_05": "reject_holm"})
    results["alpha"] = float(alpha)
    results = results.sort_values(
        ["Test"] + ALIGNMENT_GROUP_COLUMNS + ["Transition"], kind="stable"
    ).reset_index(drop=True)
    return results, matched, pd.concat(speaker_tables, ignore_index=True)

In [ ]:
# Paste this entire file into a NEW Colab code cell after the failed cell.
# It preserves ALIGNMENT_PAIRS_CSV and reruns the corrected inference.
# With a fresh session, set ALIGNMENT_PAIRS_CSV to your exported pair CSV first.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

# Run after the CREMA-D alignment cells, or set this to the previously exported
# cremad_directional_alignment_pairs.csv and run only this new section.
ALIGNMENT_PAIRS_CSV = globals().get("ALIGNMENT_PAIRS_CSV", None)
ALIGNMENT_N_SIGN_FLIPS = 19_999
ALIGNMENT_N_BOOTSTRAP = 9_999
ALIGNMENT_MAX_EXACT_SPEAKERS = 16
ALIGNMENT_STATS_SEED = 20261002
ALIGNMENT_ALPHA = 0.05

ALIGNMENT_GROUP_COLUMNS = [
    "Reference_dataset", "Shift_dataset", "Model", "Layer", "Coarse"
]
ALIGNMENT_TRANSITIONS = ["MD_TO_HI", "MD_TO_LO"]


def alignment_seed(test_key, purpose, seed):
    """Stable, separate RNG streams for every hypothesis and resampling method."""
    payload = json.dumps([int(seed), list(test_key), purpose], ensure_ascii=True)
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little")


def validate_alignment_pairs(frame):
    required = ALIGNMENT_GROUP_COLUMNS + [
        "Transition", "speaker", "utterance", "mean_subtle_alignment"
    ]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Alignment-pair table is missing columns: {missing}")
    frame = frame[required].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError("Alignment-pair table must be nonempty with no missing values")
    string_columns = [c for c in required if c != "mean_subtle_alignment"]
    for column in string_columns:
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"Empty value in {column}")
    frame["mean_subtle_alignment"] = pd.to_numeric(
        frame["mean_subtle_alignment"], errors="raise"
    ).astype(float)
    scores = frame["mean_subtle_alignment"].to_numpy()
    if not np.isfinite(scores).all() or np.any(np.abs(scores) > 1 + 1e-12):
        raise ValueError("Mean cosine alignment must be finite and in [-1, 1]")
    if not set(frame["Transition"]).issubset(ALIGNMENT_TRANSITIONS):
        raise ValueError("Only MD_TO_HI and MD_TO_LO transitions are allowed")
    pair_key = ALIGNMENT_GROUP_COLUMNS + ["speaker", "utterance", "Transition"]
    if frame.duplicated(pair_key).any():
        raise ValueError("Duplicate model/emotion/speaker/utterance/transition rows")
    return frame.sort_values(pair_key, kind="stable").reset_index(drop=True)


def holm_adjust(p_values):
    """Holm family-wise correction, valid without independence between tests."""
    p_values = np.asarray(p_values, dtype=np.float64)
    if p_values.ndim != 1 or len(p_values) == 0:
        raise ValueError("Expected a nonempty one-dimensional p-value array")
    if not np.isfinite(p_values).all() or np.any((p_values < 0) | (p_values > 1)):
        raise ValueError("Invalid p-value")
    order = np.argsort(p_values, kind="stable")
    adjusted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * p_values[order]
    )
    result = np.empty_like(p_values)
    result[order] = np.minimum(adjusted, 1.0)
    return result


def speaker_cluster_inference(
    values, speakers, test_key, *,
    n_sign_flips=ALIGNMENT_N_SIGN_FLIPS,
    n_bootstrap=ALIGNMENT_N_BOOTSTRAP,
    max_exact_speakers=ALIGNMENT_MAX_EXACT_SPEAKERS,
    seed=ALIGNMENT_STATS_SEED,
):
    """Two-sided speaker sign flips plus percentile speaker-bootstrap CI.

    The statistic remains the pair-weighted mean. Every row of a speaker
    receives the same random sign, and bootstrap draws retain whole speakers.
    Sign-flip validity requires independent speaker clusters whose score
    vectors are jointly sign-symmetric under H0; a zero mean alone is not enough.
    """
    values = np.asarray(values, dtype=np.float64)
    speakers = np.asarray(speakers)
    if values.ndim != 1 or speakers.ndim != 1 or len(values) != len(speakers):
        raise ValueError("Values and speakers must be equal-length vectors")
    if len(values) == 0 or not np.isfinite(values).all() or pd.isna(speakers).any():
        raise ValueError("Values and speakers must be nonempty and finite/nonmissing")
    if n_sign_flips < 1 or n_bootstrap < 2 or not 0 <= max_exact_speakers <= 20:
        raise ValueError("Invalid resampling counts or exact-enumeration limit")

    cluster_input = pd.DataFrame({"speaker": speakers.astype(str), "value": values})
    # Stable ordering also makes results invariant to input-row ordering.
    cluster_input = cluster_input.sort_values(["speaker", "value"], kind="stable")
    clusters = cluster_input.groupby("speaker", sort=True)["value"].agg(["sum", "count"])
    clusters = clusters.rename(columns={"sum": "score_sum", "count": "N_pairs"}).reset_index()
    n_speakers = len(clusters)
    if n_speakers < 2:
        raise ValueError(f"{test_key}: at least two independent speakers are required")

    sums = clusters["score_sum"].to_numpy(dtype=np.float64)
    counts = clusters["N_pairs"].to_numpy(dtype=np.int64)
    denominator = int(counts.sum())
    observed = float(sums.sum() / denominator)
    # Handle floating-point ties relative to the arithmetic scale, including
    # cancellation near zero. This cannot turn a null p-value into zero.
    tolerance = 100 * np.finfo(np.float64).eps * max(
        float(np.abs(sums).sum() / denominator), np.finfo(np.float64).tiny
    )

    exact = n_speakers <= max_exact_speakers
    n_draws = (1 << n_speakers) if exact else int(n_sign_flips)
    flip_seed = alignment_seed(test_key, "speaker_sign_flip", seed)
    flip_rng = np.random.default_rng(flip_seed)
    extreme = 0
    for start in range(0, n_draws, 1024):
        batch = min(1024, n_draws - start)
        if exact:
            patterns = np.arange(start, start + batch, dtype=np.uint64)[:, None]
            bits = (patterns >> np.arange(n_speakers, dtype=np.uint64)) & np.uint64(1)
            signs = 2 * bits.astype(np.int8) - 1
        else:
            signs = 2 * flip_rng.integers(0, 2, size=(batch, n_speakers), dtype=np.int8) - 1
        null_means = (signs * sums[None, :]).sum(axis=1) / denominator
        extreme += int(np.count_nonzero(np.abs(null_means) >= abs(observed) - tolerance))

    # Exact enumeration includes the observed sign pattern; do not add one.
    # Random Monte Carlo draws use the finite-resampling +1 correction.
    p_value = extreme / n_draws if exact else (extreme + 1) / (n_draws + 1)

    bootstrap_seed = alignment_seed(test_key, "speaker_bootstrap", seed)
    bootstrap_rng = np.random.default_rng(bootstrap_seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)
    for start in range(0, n_bootstrap, 1024):
        batch = min(1024, n_bootstrap - start)
        draw = bootstrap_rng.integers(0, n_speakers, size=(batch, n_speakers))
        bootstrap_means[start:start + batch] = (
            sums[draw].sum(axis=1) / counts[draw].sum(axis=1)
        )
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    clusters["speaker_mean"] = clusters["score_sum"] / clusters["N_pairs"]

    result = {
        "estimate": observed,
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "p_raw": float(p_value),
        "N_pairs": denominator,
        "N_speakers": n_speakers,
        "min_pairs_per_speaker": int(counts.min()),
        "max_pairs_per_speaker": int(counts.max()),
        "test_method": "exact_speaker_sign_flip" if exact else "monte_carlo_speaker_sign_flip",
        "alternative": "two-sided",
        "null_value": 0.0,
        "sign_patterns_evaluated": n_draws,
        "extreme_patterns": extreme,
        "bootstrap_draws": int(n_bootstrap),
        "sign_flip_seed": flip_seed,
        "bootstrap_seed": bootstrap_seed,
        "fewer_than_10_speakers": bool(n_speakers < 10),
    }
    return result, clusters


def alignment_matching_audit(frame):
    """Inventory complete and incomplete matches without changing input scores."""
    frame = validate_alignment_pairs(frame)
    match_key = ALIGNMENT_GROUP_COLUMNS + ["speaker", "utterance"]
    inventory = frame.pivot(
        index=match_key, columns="Transition", values="mean_subtle_alignment"
    ).reindex(columns=ALIGNMENT_TRANSITIONS)
    inventory.columns.name = None
    inventory = inventory.reset_index()
    has_hi = inventory["MD_TO_HI"].notna()
    has_lo = inventory["MD_TO_LO"].notna()
    inventory["match_status"] = np.select(
        [has_hi & has_lo, has_hi], ["complete", "HI_only"], default="LO_only"
    )
    # Missing counterparts stay missing. Never fill with zero or match row order.
    inventory["HI_minus_LO"] = inventory["MD_TO_HI"] - inventory["MD_TO_LO"]
    coverage_rows = []
    for keys, group in inventory.groupby(ALIGNMENT_GROUP_COLUMNS, sort=True):
        complete = group["match_status"].eq("complete")
        coverage_rows.append({
            **dict(zip(ALIGNMENT_GROUP_COLUMNS, keys)),
            "N_HI_available": int(group["MD_TO_HI"].notna().sum()),
            "N_LO_available": int(group["MD_TO_LO"].notna().sum()),
            "N_complete_pairs": int(complete.sum()),
            "N_matched_speakers": int(group.loc[complete, "speaker"].nunique()),
            "N_HI_only": int(group["match_status"].eq("HI_only").sum()),
            "N_LO_only": int(group["match_status"].eq("LO_only").sum()),
        })
    return inventory, pd.DataFrame(coverage_rows)


def directional_alignment_inference(frame, *, alpha=ALIGNMENT_ALPHA, **resampling):
    """Test all available transition scores; paired contrasts use complete matches.

    Incomplete matching never blocks the alignment-versus-zero tests. Contrasts
    with fewer than two matched speakers are explicitly marked not tested.
    Three hypotheses per observed model/layer/emotion group form ONE planned
    Holm family, including unavailable hypotheses conservatively as p=1 only
    for adjustment. Unavailable p-values remain missing in the output.
    """
    if not 0 < alpha < 1:
        raise ValueError("alpha must be between zero and one")
    frame = validate_alignment_pairs(frame)
    match_inventory, coverage = alignment_matching_audit(frame)
    matched = match_inventory.loc[match_inventory["match_status"].eq("complete")].copy()
    matched = matched.drop(columns="match_status").reset_index(drop=True)
    matched_groups = {
        keys: group for keys, group in matched.groupby(ALIGNMENT_GROUP_COLUMNS, sort=True)
    }

    results, speaker_tables = [], []

    def run_group(group, keys, test, transition, score_column):
        meta = dict(zip(ALIGNMENT_GROUP_COLUMNS, keys))
        meta.update({
            "Test": test, "Transition": transition,
            "analysis_population": (
                "all_available_transition_pairs" if test == "alignment_vs_zero"
                else "complete_HI_LO_matches_only"
            ),
        })
        n_speakers = int(group["speaker"].nunique())
        if n_speakers < 2:
            reason = (
                "not_tested_no_matched_pairs" if group.empty and test == "paired_HI_minus_LO"
                else "not_tested_no_rows" if group.empty
                else "not_tested_fewer_than_2_speakers"
            )
            results.append({
                **meta, "status": reason,
                "estimate": float(group[score_column].mean()) if len(group) else np.nan,
                "CI95_low": np.nan, "CI95_high": np.nan, "p_raw": np.nan,
                "N_pairs": len(group), "N_speakers": n_speakers,
                "test_method": "not_tested", "alternative": "two-sided", "null_value": 0.0,
                "sign_patterns_evaluated": 0, "bootstrap_draws": 0,
                "fewer_than_10_speakers": bool(n_speakers < 10),
            })
            return
        result, speakers = speaker_cluster_inference(
            group[score_column].to_numpy(),
            group["speaker"].to_numpy(),
            tuple(keys) + (test, transition),
            **resampling,
        )
        results.append({**meta, "status": "tested", **result})
        for name, value in meta.items():
            speakers[name] = value
        speaker_tables.append(speakers)

    for keys, group in frame.groupby(ALIGNMENT_GROUP_COLUMNS, sort=True):
        for transition in ALIGNMENT_TRANSITIONS:
            available = group.loc[group["Transition"].eq(transition)]
            run_group(available, keys, "alignment_vs_zero", transition, "mean_subtle_alignment")
        complete = matched_groups.get(keys, matched.iloc[:0])
        run_group(complete, keys, "paired_HI_minus_LO", "MD_TO_HI_minus_MD_TO_LO", "HI_minus_LO")

    result_rows = results
    results = pd.DataFrame(result_rows)
    # Nullable uint64 retains large seeds exactly even when skipped rows have no seed.
    for seed_column in ["sign_flip_seed", "bootstrap_seed"]:
        results[seed_column] = pd.array(
            [row.get(seed_column, pd.NA) for row in result_rows], dtype="UInt64"
        )
    tested = results["status"].eq("tested")
    results["p_holm"] = holm_adjust(results["p_raw"].fillna(1.0).to_numpy())
    results.loc[~tested, "p_holm"] = np.nan
    results["holm_family_size"] = len(results)
    results["N_tests_performed"] = int(tested.sum())
    results["reject_holm_0_05"] = pd.array(results["p_holm"] <= alpha, dtype="boolean")
    results.loc[~tested, "reject_holm_0_05"] = pd.NA
    # The column is explicitly named by the configured threshold below.
    if alpha != 0.05:
        results = results.rename(columns={"reject_holm_0_05": "reject_holm"})
    results["alpha"] = float(alpha)
    results = results.merge(coverage, on=ALIGNMENT_GROUP_COLUMNS, how="left", validate="many_to_one")
    results = results.sort_values(
        ["Test"] + ALIGNMENT_GROUP_COLUMNS + ["Transition"], kind="stable"
    ).reset_index(drop=True)
    speakers = (
        pd.concat(speaker_tables, ignore_index=True) if speaker_tables
        else pd.DataFrame(columns=["speaker", "score_sum", "N_pairs", "speaker_mean"]
                          + ALIGNMENT_GROUP_COLUMNS + ["Test", "Transition", "analysis_population"])
    )
    return results, matched, speakers


if ALIGNMENT_PAIRS_CSV is not None:
    _alignment_csv = Path(ALIGNMENT_PAIRS_CSV).expanduser()
    _alignment_input = pd.read_csv(
        _alignment_csv,
        dtype={c: str for c in ALIGNMENT_GROUP_COLUMNS + ["speaker", "utterance", "Transition"]},
    )
    _alignment_source = str(_alignment_csv.resolve())
elif "directional_pairs" in globals():
    _alignment_input = directional_pairs.copy()
    _alignment_source = "in-memory directional_pairs from the CREMA-D alignment section"
else:
    raise RuntimeError(
        "Run the CREMA-D directional-alignment cells first, or set "
        "ALIGNMENT_PAIRS_CSV to cremad_directional_alignment_pairs.csv "
        "in the preceding new cell. Summary means/SDs cannot recover speaker clustering."
    )

alignment_statistics, alignment_matched_pairs, alignment_speaker_scores = (
    directional_alignment_inference(_alignment_input)
)
alignment_match_inventory, alignment_matching_coverage = alignment_matching_audit(_alignment_input)

if "DIRECTION_RUN_DIR" in globals():
    _alignment_parent = Path(DIRECTION_RUN_DIR)
elif ALIGNMENT_PAIRS_CSV is not None:
    _alignment_parent = Path(ALIGNMENT_PAIRS_CSV).expanduser().resolve().parent
elif "RUN_DIR" in globals():
    _alignment_parent = Path(RUN_DIR)
else:
    _alignment_parent = Path(globals().get("OUTPUT_BASE", Path.cwd() / "geometry_unified"))

_alignment_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ALIGNMENT_STATS_DIR = _alignment_parent / (
    f"alignment_statistics_{_alignment_stamp}_{uuid.uuid4().hex[:8]}"
)
ALIGNMENT_STATS_DIR.mkdir(parents=True, exist_ok=False)
_alignment_clean_input = validate_alignment_pairs(_alignment_input)
_alignment_clean_input.to_csv(ALIGNMENT_STATS_DIR / "alignment_test_input_pairs.csv", index=False)
alignment_statistics.to_csv(ALIGNMENT_STATS_DIR / "directional_alignment_statistics.csv", index=False)
alignment_matched_pairs.to_csv(ALIGNMENT_STATS_DIR / "directional_alignment_matched_pairs.csv", index=False)
alignment_speaker_scores.to_csv(ALIGNMENT_STATS_DIR / "directional_alignment_speaker_scores.csv", index=False)
alignment_match_inventory.to_csv(ALIGNMENT_STATS_DIR / "directional_alignment_match_inventory.csv", index=False)
alignment_matching_coverage.to_csv(ALIGNMENT_STATS_DIR / "directional_alignment_matching_coverage.csv", index=False)

_alignment_input_hash = hashlib.sha256(
    (ALIGNMENT_STATS_DIR / "alignment_test_input_pairs.csv").read_bytes()
).hexdigest()
alignment_statistics_manifest = {
    "created_utc": _alignment_stamp,
    "input_source": _alignment_source,
    "exported_input_sha256": _alignment_input_hash,
    "statistic": "pair-weighted mean cosine alignment, or mean matched HI-minus-LO difference",
    "null_hypotheses": ["mean alignment = 0 per transition", "mean matched HI-minus-LO difference = 0"],
    "alternative": "two-sided absolute mean",
    "resampling_unit": "speaker; all rows from each speaker move together",
    "sign_flip_assumption": (
        "independent speaker clusters; within each tested condition, the cluster score vector "
        "is jointly symmetric under sign reversal under H0. A zero mean alone is insufficient. "
        "For the paired contrast this is symmetry of the within-speaker HI-minus-LO differences."
    ),
    "conditioning": "EmoS-derived reference directions are fixed; their estimation uncertainty is not resampled",
    "monte_carlo_sign_flips": ALIGNMENT_N_SIGN_FLIPS,
    "exact_enumeration_if_speakers_at_most": ALIGNMENT_MAX_EXACT_SPEAKERS,
    "p_value": "extreme/all_patterns for exact enumeration; (extreme+1)/(draws+1) for Monte Carlo",
    "bootstrap_draws": ALIGNMENT_N_BOOTSTRAP,
    "bootstrap_interval": "pointwise 95% percentile CI; whole-speaker resampling; pair-weighted ratio of sums",
    "multiple_testing": (
        "Holm FWER across all planned hypotheses as one family; unavailable hypotheses "
        "use p=1 internally for adjustment and remain missing/not tested in the exported table"
    ),
    "family_size": len(alignment_statistics),
    "tests_performed": int(alignment_statistics["status"].eq("tested").sum()),
    "alpha": ALIGNMENT_ALPHA,
    "seed": ALIGNMENT_STATS_SEED,
    "matching": "same reference dataset, shift dataset, model, layer, coarse emotion, speaker and utterance",
    "incomplete_matching": (
        "all available rows remain in alignment-vs-zero tests; paired contrasts use only "
        "complete matches, target that subset, and are not tested with fewer than two matched speakers"
    ),
    "python": __import__("platform").python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
(ALIGNMENT_STATS_DIR / "directional_alignment_statistics_manifest.json").write_text(
    json.dumps(alignment_statistics_manifest, indent=2), encoding="utf-8"
)

try:
    from IPython.display import display as _alignment_display
except ImportError:
    _alignment_display = print

_alignment_display_columns = [
    "Model", "Layer", "Coarse", "Transition", "status", "estimate", "CI95_low", "CI95_high",
    "N_pairs", "N_speakers", "p_raw", "p_holm", "reject_holm_0_05", "test_method"
]
if ALIGNMENT_ALPHA != 0.05:
    _alignment_display_columns = [
        "reject_holm" if c == "reject_holm_0_05" else c for c in _alignment_display_columns
    ]
for _alignment_test in ["alignment_vs_zero", "paired_HI_minus_LO"]:
    print(_alignment_test)
    _alignment_display(alignment_statistics.loc[
        alignment_statistics["Test"].eq(_alignment_test), _alignment_display_columns
    ].round(6))

print("HI/LO matching coverage (all rows remain in the alignment-versus-zero tests):")
_alignment_display(alignment_matching_coverage)
print(
    f"Holm family: {len(alignment_statistics)} planned tests; "
    f"{alignment_statistics['status'].eq('tested').sum()} performed, alpha={ALIGNMENT_ALPHA}. "
    "CI95 columns are pointwise intervals, not simultaneous intervals."
)
if alignment_statistics["status"].ne("tested").any():
    print("Not-tested rows have missing p-values; inspect status and matching coverage. They are not nonsignificant results.")
if (alignment_statistics["status"].eq("tested") & alignment_statistics["fewer_than_10_speakers"]).any():
    print("Some comparisons contain fewer than 10 independent speakers; inspect N_speakers and coarse exact p-value resolution.")
print("Saved directional-alignment inference:", ALIGNMENT_STATS_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')